<a href="https://colab.research.google.com/github/Sak-shi437/deep_learning/blob/main/mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt



# 1. LOAD & PREPROCESS DATA

In [ ]:
print("loading MNIST")
mnist = fetch_openml('mnist_784', version=1, as_frame = False)
X, y = mnist.data, mnist.target.astype(int)

loading MNIST


In [ ]:
# Normalize pixel values from [0, 255] to [0, 1]
X = X / 255.0

In [ ]:

# One-hot encode labels: e.g. label 3 -> [0,0,0,1,0,0,0,0,0,0]


def one_hot(y, num_classes=10):
  encoded = np.zeros((y.size, num_classes))
  encoded[np.arange(y.size), y] = 1
  return encoded

In [ ]:
Y = one_hot(y)


# Split into train/test, then transpose so each COLUMN is one example.
# Shapes: X -> (784, num_examples), Y -> (10, num_examples)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.15, random_state=42)
X_train, X_test = X_train.T, X_test.T
Y_train, Y_test = Y_train.T, Y_test.T

print(f"Train set: {X_train.shape}, Test set: {X_test.shape}")


Train set: (784, 59500), Test set: (784, 10500)


 2. INITIALIZE WEIGHTS & BIASES

In [ ]:
def init_params(input_size=784, hidden_size=128,output_size=10):
  # He initialization (good pairing with ReLU)
  W1= np.random.randn(hidden_size, input_size)* np.sqrt(2. / input_size)
  b1 = np.zeros((hidden_size, 1))
  W2 = np.random.randn(output_size, hidden_size)* np.sqrt(2. / hidden_size)

  b2= np.zeros((output_size, 1))
  return W1, b1, W2, b2

# 3. ACTIVATION FUNCTIONS


In [ ]:
def relu(Z):
  return np.maximum(0, Z)


def relu_derivative(Z):
  return (Z > 0).astype(float)


def softmax(Z):
   # Subtract max for numerical stability (prevents overflow in exp)
   expZ = np.exp(Z - np.max(Z, axis= 0, keepdims=True))
   return expZ / np.sum(expZ, axis=0, keepdims= True)



4. FORWARD PROPAGATION

In [ ]:
def forward_propagation(X, W1, b1, W2, b2):
  Z1 = W1.dot(X) + b1
  A1 = relu(Z1)
  Z2 = W2.dot(A1) + b2
  A2 = softmax(Z2)
  cache = (Z1,A1,Z2,A2)
  return A2,cache

# 5. LOSS FUNCTION (Categorical Cross-Entropy)

In [ ]:
def compute_loss(A2, Y):
  m = Y.shape[1]
  epsilon = 1e-8  # avoid log(0)
  loss = -np.sum(Y * np.log(A2 + epsilon)) / m
  return loss

# 6. BACKWARD PROPAGATION

In [ ]:
def backward_propagation(X, Y, W2, cache):
    Z1, A1, Z2, A2 = cache
    m = X.shape[1]

    # Output layer gradients (softmax + cross-entropy simplifies beautifully)
    dZ2 = A2 - Y                              # (10, m)
    dW2 = (dZ2 @ A1.T) / m                    # (10, 128)
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m

    # Hidden layer gradients (chain rule back through W2 and ReLU)
    dA1 = W2.T @ dZ2                          # (128, m)
    dZ1 = dA1 * relu_derivative(Z1)           # (128, m)
    dW1 = (dZ1 @ X.T) / m                     # (128, 784)
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    return dW1, db1, dW2, db2

# 7. UPDATE WEIGHTS (Gradient Descent)

In [ ]:
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, lr):
  W1 = W1 - lr * dW1
  b1 = b1 - lr * db1
  W2 = W2 - lr * dW2
  b2 = b2 - lr * db2
  return W1, b1, W2, b

# 8. ACCURACY

In [ ]:
def compute_accuracy(A2, Y):
    predictions = np.argmax(A2, axis=0)
    labels = np.argmax(Y, axis=0)
    return np.mean(predictions == labels)

# 9. TRAINING LOOP

In [ ]:
def train(X_train, Y_train, X_test, Y_test, hidden_size=128, epochs=200, lr=0.1):
    W1, b1, W2, b2 = init_params(hidden_size=hidden_size)
    train_losses, train_accs, test_accs = [], [], []

    for epoch in range(epochs):
        # Forward pass
        A2, cache = forward_propagation(X_train, W1, b1, W2, b2)

        # Loss + accuracy
        loss = compute_loss(A2, Y_train)
        acc = compute_accuracy(A2, Y_train)

        # Backward pass
        dW1, db1, dW2, db2 = backward_propagation(X_train, Y_train, W2, cache)

        # Update weights
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, lr)

        train_losses.append(loss)
        train_accs.append(acc)

        if epoch % 10 == 0 or epoch == epochs - 1:
            A2_test, _ = forward_propagation(X_test, W1, b1, W2, b2)
            test_acc = compute_accuracy(A2_test, Y_test)
            test_accs.append(test_acc)
            print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Train Acc: {acc:.4f} | Test Acc: {test_acc:.4f}")

    return W1, b1, W2, b2, train_losses, train_accs, test_accs

# End of train function


# 10. RUN TRAINING


In [ ]:
if __name__ == "__main__":


  W1, b1, W2, b2, train_losses, train_accs, test_accs = train(X_train, Y_train, X_test, Y_test, hidden_size=128, epochs=200, lr=0.5)

    # Final test accuracy
  A2_test, _ = forward_propagation(X_test, W1, b1, W2, b2)
  final_acc = compute_accuracy(A2_test, Y_test)
  print(f"\nFinal Test Accuracy: {final_acc:.4f}")

    # Plot loss curve
  plt.figure(figsize=(10, 4))
  plt.subplot(1, 2, 1)
  plt.plot(train_losses)
  plt.title("Training Loss")
  plt.xlabel("Epoch")
  plt.ylabel("Loss")

  plt.subplot(1, 2, 2)
  plt.plot(train_accs, label="Train Accuracy")
  plt.title("Training Accuracy")
  plt.xlabel("Epoch")
  plt.ylabel("Accuracy")
  plt.legend()

  plt.tight_layout()
  plt.savefig("training_curves.png")
  print("Saved training_curves.png")

NameError: name 'forward_propagation' is not defined